# HiGAN+ Experiment 1: Vertical Random Masking

This notebook trains HiGAN+ with **vertical stripe masking** applied to generated images during training.

## Motivation
Vertical masking encourages the generator to:
- Learn continuity across character strokes
- Avoid over-reliance on rigid character boundaries
- Improve local spacing consistency
- Learn more robust sequential handwriting structure

## Implementation
Random vertical stripe masks are applied to `fake_imgs`, `style_imgs`, and `recn_imgs` during generator training only.

## 1. Setup Environment

In [ ]:
# Install dependencies
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install tensorboard munch distance h5py pillow matplotlib numpy tqdm pyyaml

## 2. Clone Repository

In [ ]:
import os

# Clone the repository with masking branch
if os.path.exists('/kaggle/working/higanplus'):
    print("Repository already exists, pulling latest changes...")
    %cd /kaggle/working/higanplus
    !git pull origin masking
else:
    print("Cloning repository...")
    %cd /kaggle/working
    !git clone -b masking https://github.com/nguyenthinhucuynh/higanplus.git

%cd /kaggle/working/higanplus/HiGAN+

## 3. Download IAM Dataset

In [ ]:
# Create data directory
!mkdir -p data/iam

# Download IAM dataset (processed h5py files)
!wget -O data/iam/trnvalset_words64_OrgSz.hdf5 https://github.com/ganji15/HiGANplus/releases/download/dataset/trnvalset_words64_OrgSz.hdf5
!wget -O data/iam/testset_words64_OrgSz.hdf5 https://github.com/ganji15/HiGANplus/releases/download/dataset/testset_words64_OrgSz.hdf5

print("Dataset downloaded successfully!")

## 4. Verify Configuration

In [ ]:
# Check the vertical masking config
!cat configs/gan_iam_vertical_masking.yml | grep -A 5 "masking_mode"

## 5. Start Training

In [ ]:
# Train with vertical masking
!python train.py --config ./configs/gan_iam_vertical_masking.yml

## 6. Monitor Training (Optional)

You can monitor training progress using TensorBoard:

In [ ]:
# Load TensorBoard extension
%load_ext tensorboard
%tensorboard --logdir=./runs

## 7. Evaluation (After Training)

After training completes, evaluate the model:

In [ ]:
# Find the latest checkpoint
import os
import glob

run_dirs = glob.glob('./runs/gan_iam_vertical_masking-*')
if run_dirs:
    latest_run = sorted(run_dirs)[-1]
    ckpt_path = os.path.join(latest_run, 'ckpts/best.pth')
    print(f"Latest checkpoint: {ckpt_path}")
else:
    print("No checkpoints found")

In [ ]:
# Quantitative evaluation
!python test.py --config ./configs/gan_iam_vertical_masking.yml --ckpt {ckpt_path} --guided True

In [ ]:
# Qualitative evaluation - Reference-guided synthesis
!python eval_demo.py --config ./configs/gan_iam_vertical_masking.yml --ckpt {ckpt_path} --mode style

## 8. Download Results

Download the trained model and generated samples:

In [ ]:
# Zip the results
!zip -r vertical_masking_results.zip ./runs/gan_iam_vertical_masking-*/

# Download using Kaggle's file download
from IPython.display import FileLink
FileLink('vertical_masking_results.zip')